# Step 5: Intersection of Personalized EEG Channels

This notebook performs the fifth and final project step: identify the EEG channels retained by **every one of the 14 patient-specific models**.

The personalized models are upstream of this notebook. This step does not train a model or choose `k`; it only computes the strict set intersection of their binary channel-selection outputs.

## Required input contract

The function expects one wide `pandas.DataFrame`:

- one row per patient;
- a `patient_id` column;
- one column per harmonized EEG channel;
- `1` if that patient's personalized model retained the channel;
- `0` if it did not retain the channel (including a channel that is unavailable for that patient).

Channel labels must be standardized before this step. For example, `EEG Fp1`, `Fp1-REF`, and `FP1` must not remain as separate columns if they represent the same electrode.

A strict intersection can be empty. That is a valid result and should not be replaced by a majority-vote set.

In [1]:
from collections.abc import Sequence

import pandas as pd


PROJECT_PATIENT_IDS = (
    "PN00", "PN01", "PN03", "PN05", "PN06", "PN07", "PN09",
    "PN10", "PN11", "PN12", "PN13", "PN14", "PN16", "PN17",
)

In [2]:
def find_channel_intersection(
    channel_selections: pd.DataFrame,
    *,
    patient_col: str = "patient_id",
    expected_patient_ids: Sequence[str] | None = PROJECT_PATIENT_IDS,
) -> list[str]:
    """Return channels selected by every patient-specific model.

    Parameters
    ----------
    channel_selections:
        Wide binary table with one row per patient, a patient identifier
        column, and one 0/1 column per harmonized EEG channel.
    patient_col:
        Name of the patient identifier column.
    expected_patient_ids:
        Patients that must be represented exactly once. The project defaults
        to all 14 Siena patient IDs. Pass None only for a deliberate subset.

    Returns
    -------
    list[str]
        Channel names whose value is 1 for every included patient. Original
        DataFrame column order is preserved.
    """
    if not isinstance(channel_selections, pd.DataFrame):
        raise TypeError("channel_selections must be a pandas DataFrame")
    if patient_col not in channel_selections.columns:
        raise ValueError(f"Missing patient identifier column: {patient_col!r}")
    if channel_selections.empty:
        raise ValueError("channel_selections cannot be empty")
    if channel_selections[patient_col].isna().any():
        raise ValueError("patient IDs cannot be missing")

    patient_ids = channel_selections[patient_col].astype(str)
    duplicated = patient_ids[patient_ids.duplicated()].unique().tolist()
    if duplicated:
        raise ValueError(f"Each patient must appear once; duplicates: {duplicated}")

    if expected_patient_ids is not None:
        expected = {str(patient_id) for patient_id in expected_patient_ids}
        observed = set(patient_ids)
        missing = sorted(expected - observed)
        unexpected = sorted(observed - expected)
        if missing or unexpected:
            raise ValueError(
                f"Patient mismatch. Missing: {missing}; unexpected: {unexpected}"
            )

    channel_cols = [
        column for column in channel_selections.columns if column != patient_col
    ]
    if not channel_cols:
        raise ValueError("No channel columns were provided")

    binary = channel_selections[channel_cols]
    invalid_mask = ~binary.isin([0, 1])
    if invalid_mask.any().any():
        invalid_columns = invalid_mask.any(axis=0)
        bad = invalid_columns.index[invalid_columns].tolist()
        raise ValueError(
            f"Channel selections must contain only 0 or 1; check: {bad}"
        )

    selected_by_every_patient = binary.eq(1).all(axis=0)
    return selected_by_every_patient.index[selected_by_every_patient].tolist()

## Worked example

This synthetic table demonstrates the handoff format. Replace it with the personalized-model output when that pipeline is ready.

In [3]:
example_selections = pd.DataFrame(
    {
        "patient_id": PROJECT_PATIENT_IDS,
        "Fp1": [1] * 14,
        "Fp2": [1] * 13 + [0],
        "C3": [1] * 14,
        "C4": [0, 1] * 7,
        "O1": [1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1],
    }
)

intersection_channels = find_channel_intersection(example_selections)

display(example_selections)
print("Channels selected by all 14 patients:", intersection_channels)
assert intersection_channels == ["Fp1", "C3"]

,patient_id,Fp1,Fp2,C3,C4,O1
0,PN00,1,1,1,0,1
1,PN01,1,1,1,1,1
2,PN03,1,1,1,0,0
3,PN05,1,1,1,1,1
4,PN06,1,1,1,0,1
5,PN07,1,1,1,1,0
6,PN09,1,1,1,0,1
7,PN10,1,1,1,1,1
8,PN11,1,1,1,0,0
9,PN12,1,1,1,1,1


Channels selected by all 14 patients: ['Fp1', 'C3']


## Use with the personalized-model output

When the upstream table exists, load it and call the same function. Keep this cell commented until the final filename is agreed upon.

In [4]:
# personalized_channel_selections = pd.read_csv(
#     "../results/personalized_channel_selections.csv"
# )
# common_channels = find_channel_intersection(personalized_channel_selections)
# print(f"Strict intersection ({len(common_channels)} channels): {common_channels}")

## Interpretation

The returned list is the strict mathematical intersection:

$$\mathcal{C}_{common} = \bigcap_{p=1}^{14} \mathcal{C}_p$$

A channel appears in the result only if every personalized model retained it. This describes selection agreement; by itself, it does **not** establish physiological importance, causal relevance, or out-of-sample predictive performance.